# DSWP audio to WhAM embeddings

This notebook is a reproducible starter pipeline for public Dominica Sperm Whale Project audio. It downloads a small named sample, extracts official WhAM layer embeddings when local checkpoints are present, and explores the resulting representation with PCA, UMAP or t-SNE and KMeans.

The public dataset has audio only. Its card reports 1,501 WAV files, no per-file metadata and one vocal clan. The notebook therefore does not invent social-unit labels. The supervised section is gated on a separately verified annotation CSV.

Sources: [WhAM](https://github.com/Project-CETI/wham), [paper](https://arxiv.org/abs/2512.02206), [DSWP](https://huggingface.co/datasets/orrp/DSWP), [weights](https://doi.org/10.5281/zenodo.17633708).

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'vampnet').is_dir() and (candidate / 'tutorial').is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the WhAM repository')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tutorial.wham_dswp import (
    audio_manifest,
    build_wham_interface,
    cluster_embeddings,
    download_dswp_audio,
    extract_embeddings,
    fit_label_baseline,
    fit_social_unit_baseline,
    load_annotation_labels,
    load_embedding_artifact,
    reduce_embeddings,
    save_embedding_artifact,
)

DATA_DIR = PROJECT_ROOT / 'work' / 'dswp_audio'
ARTIFACT_DIR = PROJECT_ROOT / 'work' / 'artifacts'
COARSE_CHECKPOINT = PROJECT_ROOT / 'vampnet' / 'models' / 'coarse.pth'
CODEC_CHECKPOINT = PROJECT_ROOT / 'vampnet' / 'models' / 'codec.pth'
EMBEDDING_ARTIFACT = ARTIFACT_DIR / 'dswp_wham_embeddings.npz'
SAMPLE_FILES = ['1.wav', '10.wav', '100.wav']
LAYER = 10
DEVICE = 'cpu'


## 1. Download and inspect public DSWP audio

Only the named files are downloaded. Increase `SAMPLE_FILES` after the smoke run. The numeric filenames are not social-unit labels.

In [ ]:
audio_paths = download_dswp_audio(SAMPLE_FILES, DATA_DIR)
manifest = audio_manifest(audio_paths)
display(manifest)
print(f'Downloaded {len(audio_paths)} audio files into {DATA_DIR}')

## 2. Extract official WhAM embeddings

WhAM checkpoints are large. The official weights record is about 3.1 GB in total. This notebook needs `coarse.pth` and `codec.pth` for embeddings and deliberately does not require `c2f.pth` or `wavebeat.pth`.

The official model path returns layerwise activations with dimension 1280. We mean-pool time and keep the selected layer. If the checkpoints are absent, this cell stops at an explicit message rather than substituting a different model.

In [ ]:
def usable_checkpoint(path):
    return path.is_file() and path.stat().st_size > 1_000_000

weights_ready = usable_checkpoint(COARSE_CHECKPOINT) and usable_checkpoint(CODEC_CHECKPOINT)
if not weights_ready:
    print('WhAM checkpoints are not present.')
    print(f'Download coarse.pth and codec.pth into {COARSE_CHECKPOINT.parent}')
    print('The rest of the notebook remains valid once the artifact exists.')
else:
    interface = build_wham_interface(
        PROJECT_ROOT,
        COARSE_CHECKPOINT,
        CODEC_CHECKPOINT,
        device=DEVICE,
    )
    vectors = extract_embeddings(audio_paths, interface, layer=LAYER)
    layerwise_shape = (len(audio_paths), getattr(interface.coarse, 'n_layers', 'unknown'), vectors.shape[-1])
    print(f'Selected layer vectors: {vectors.shape}')
    save_embedding_artifact(
        EMBEDDING_ARTIFACT,
        audio_paths,
        vectors[:, None, :],
        layer=LAYER,
    )
    print(f'Saved {EMBEDDING_ARTIFACT}')

## 3. PCA, clustering and visualization

The exploratory metrics below describe geometry in the selected embedding layer. They do not identify whales or prove social-unit structure. For UMAP, use more than the three-file smoke sample.

In [ ]:
if not EMBEDDING_ARTIFACT.exists():
    print('No WhAM artifact yet. Run the extraction cell after installing the official checkpoints.')
else:
    artifact = load_embedding_artifact(EMBEDDING_ARTIFACT)
    vectors = artifact['embeddings'][:, 0, :]
    projection = reduce_embeddings(vectors, method='pca')
    n_clusters = min(3, max(2, len(vectors) // 2))
    clustering = cluster_embeddings(vectors, n_clusters=n_clusters)
    plot_data = pd.DataFrame({
        'file': [path.name for path in artifact['audio_paths']],
        'x': projection[:, 0],
        'y': projection[:, 1],
        'cluster': clustering['labels'].astype(str),
    })
    print(f"KMeans silhouette: {clustering['silhouette']}")
    import plotly.express as px
    fig = px.scatter(plot_data, x='x', y='y', color='cluster', hover_name='file', title='WhAM layer projection')
    fig.show()

## 4. Optional classification with recording groups

The public release has no verified `Unit`, `coda_label` or recording provenance. Supply a separate CSV with `file`, `recording_group` and at least one label column. The group must identify the source recording or a larger independent unit that prevents related snippets crossing the split. Verify provenance independently. A group split alone does not establish independence or reproduce the paper. Both classes and sufficient independent groups are needed.


In [ ]:
ANNOTATION_CSV = PROJECT_ROOT / 'data' / 'annotations' / 'dswp_unit_annotations.csv'
if not EMBEDDING_ARTIFACT.exists() or not ANNOTATION_CSV.exists():
    print('Supervised sections skipped because verified annotations are unavailable.')
else:
    artifact = load_embedding_artifact(EMBEDDING_ARTIFACT)
    columns = set(pd.read_csv(ANNOTATION_CSV, nrows=0).columns)
    by_path = {path.name: index for index, path in enumerate(artifact['audio_paths'])}
    for label_column in ['Unit', 'coda_label']:
        if label_column not in columns:
            print(f'{label_column} skipped because annotations are unavailable.')
            continue
        matched_paths, labels = load_annotation_labels(
            ANNOTATION_CSV, artifact['audio_paths'], label_column=label_column
        )
        group_paths, groups = load_annotation_labels(
            ANNOTATION_CSV, matched_paths, label_column='recording_group'
        )
        if group_paths != matched_paths:
            raise ValueError('Each labeled file needs a verified recording group')
        indices = [by_path[path.name] for path in matched_paths]
        result = fit_label_baseline(artifact['embeddings'][indices, 0, :], labels, groups)
        print(f'{label_column} group-held-out report')
        print(result['report'])


## Interpretation boundary

A successful embedding and a visually separated cluster are not evidence of social units. The public files are isolated snippets without behavioral context and come from one vocal clan. A publishable classification claim needs verified annotations, a leakage-safe split, a baseline, confidence intervals and an explicit account of recording provenance.